In [1]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# =========================
# Read Files
# =========================

nse_hist = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("Files/data/raw/nse_trade_hist/*.csv")
)

nse_inc = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("Files/data/raw/nse_trade_inc/*.csv")
)

bse_hist = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("Files/data/raw/bse_trade_hist/*.csv")
)

bse_inc = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("Files/data/raw/bse_trade_inc/*.csv")
)

# =========================
# Standardize Function
# =========================

def standardize_trade(df, source):

    return (
        df
        .withColumnRenamed("Date", "date")
        .withColumnRenamed("Symbol", "symbol")
        .withColumnRenamed("Open", "open")
        .withColumnRenamed("High", "high")
        .withColumnRenamed("Low", "low")
        .withColumnRenamed("Close", "close")
        .withColumnRenamed("Volume", "volume")

        .select(
            "date",
            "symbol",
            "open",
            "high",
            "low",
            "close",
            "volume"
        )

        .withColumn("date", F.to_date("date"))

        .withColumn("open", F.col("open").cast("double"))
        .withColumn("high", F.col("high").cast("double"))
        .withColumn("low", F.col("low").cast("double"))
        .withColumn("close", F.col("close").cast("double"))
        .withColumn("volume", F.col("volume").cast("long"))

        .withColumn("source", F.lit(source))
    )

# =========================
# Standardize Data
# =========================

nse_hist = standardize_trade(nse_hist, "NSE")
nse_inc = standardize_trade(nse_inc, "NSE")

bse_hist = standardize_trade(bse_hist, "BSE")
bse_inc = standardize_trade(bse_inc, "BSE")

# =========================
# Combine Data
# =========================

trade_df = (
    nse_hist
    .unionByName(nse_inc)
    .unionByName(bse_hist)
    .unionByName(bse_inc)
)

# =========================
# Clean
# =========================

bad_rows = trade_df.filter(
    (F.col("open") <= 0) |
    (F.col("high") <= 0) |
    (F.col("low") <= 0) |
    (F.col("close") <= 0)
).count()

print(
    "Bad rows removed:",
    bad_rows
)

trade_df = trade_df.dropna(
    subset=[
        "open",
        "high",
        "low",
        "close"
    ]
)

trade_df = trade_df.filter(
    (F.col("high") >= F.col("open")) &
    (F.col("high") >= F.col("close")) &
    (F.col("low") <= F.col("open")) &
    (F.col("low") <= F.col("close")) &
    (F.col("high") >= F.col("low"))
)

trade_df = trade_df.filter(
    (F.col("open") > 0) &
    (F.col("high") > 0) &
    (F.col("low") > 0) &
    (F.col("close") > 0)
)

trade_df = trade_df.dropDuplicates()

# =========================
# NSE Priority
# =========================

trade_df = trade_df.withColumn(
    "priority",
    F.when(
        F.col("source") == "NSE",
        1
    ).otherwise(2)
)

window_spec = (
    Window
    .partitionBy(
        "date",
        "symbol"
    )
    .orderBy(
        "priority"
    )
)

price_df = (
    trade_df
    .withColumn(
        "rn",
        F.row_number().over(window_spec)
    )
    .filter(
        F.col("rn") == 1
    )
    .select(
        "date",
        "symbol",
        "open",
        "high",
        "low",
        "close"
    )
)

# =========================
# Sum NSE + BSE Volumes
# =========================

volume_df = (
    trade_df
    .groupBy(
        "date",
        "symbol"
    )
    .agg(
        F.sum("volume")
        .alias("volume")
    )
)

# =========================
# Final Table
# =========================

trade_final = (
    price_df
    .join(
        volume_df,
        ["date", "symbol"]
    )
)

# =========================
# Replace Existing Table
# =========================

spark.sql(
    "DROP TABLE IF EXISTS trade_daily"
)

# =========================
# Save Delta
# =========================

(
    trade_final
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("trade_daily")
)

# =========================
# Validation
# =========================

print("SUCCESS")
print("Rows:", trade_final.count())

display(
    trade_final.limit(10)
)

StatementMeta(, 3679aac6-83cc-44a5-97d5-25227fd98237, 3, Finished, Available, Finished, False)

Bad rows removed: 17036
SUCCESS
Rows: 14573650


SynapseWidget(Synapse.DataFrame, 378fd0ec-4395-4dd0-a65c-037002191344)